In [ ]:
# -*- coding: utf-8 -*-
"""modern_robotics_ch03.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Cih7o2ekJJLeffvXq-nMaM6di3Wzc_yj
"""


In [ ]:
# 3.1 회전 행렬 R의 역행렬을 구하는 함수
# 입력: R (3x3 matrix)
# 출력: R^T or R_inv (3x3 matrix)

import numpy as np

def RotInv(R):
  invR = np.array(R).T

  return invR


In [ ]:
# 3.2 회전축 omega의 3x3 반대칭 행렬 (skew-symmetric)을 구하는 함수
# 입력: omega (3x1 vector)
# 출력: [omega] so(3) matrix

import numpy as np

def Vec2so3(omega):
    w1, w2, w3 = np.array(omega).flatten()
    so3mat = np.array([[0, -w3, w2],
                    [w3, 0, -w1],
                    [-w2, w1, 0]])
    return so3mat


In [ ]:
# 3.3 3x3 반대칭 행렬인 so3mat를 받아 3차원 벡터 omega를 반환하는 함수
# 입력: [w] 3x3 (so3mat)
# 출력:  w  3x1 (omega)

def so32Vec(so3mat):

    omega = np.array([so3mat[2, 1], so3mat[0, 2], so3mat[1, 0]])

    return omega


In [ ]:
# 3.4 회전 expc3의 지수 좌표인 3차원 벡터 (hat_omega * theta) 로부터
# 회전축과 회전각 추출

# 입력: expc3 3x1
# 출력: hat_omega 3x1 단위축 벡터, theta 회전각 [rad]

def AxisAng3(expc3):
    theta = np.linalg.norm(expc3)

    if theta < 1e-6:
        return np.array([0, 0, 0]), theta
    else:
        hat_omega = np.array(expc3) / theta
        return hat_omega, theta


In [ ]:
# 3.5 so3mat 즉, so(3)의 행렬지수에 해당하는 회전행렬 R -> SO(3)을 계산
# 입력: hat_omega (unit rotation axis), theta (rotation angle [rad])
# 출력: R -> SO(3) (Lie-group 3x3 rotation matrix)

def MatrixExp3(hat_omega, theta):
    I = np.eye(3)
    so3omega = Vec2so3(hat_omega)

    # 로드리게스 공식: R = I + sin(theta)*[w] + (1 - cos(theta))*[w]^2
    R = I + np.sin(theta) * so3omega + (1 - np.cos(theta)) * np.dot(so3omega, so3omega)

    return R


In [ ]:
# 3.6 회전행렬 R -> SO(3)의 행렬 로그 so3mat -> so(3)를 계산
# 입력: R 3x3
# 출력: 행렬로그 so3mat 3x3
def MatrixLog3(R):
    theta = np.arccos((np.trace(R) - 1) / 2.0)
    so3mat = theta / (2 * np.sin(theta)) * (R - np.array(R).T)

    return so3mat


In [ ]:
# 3.7 회전행렬 R -> SO(3), 위치 벡터 p -> R^3로 동차 변환행렬 T를 계산
# 입력: R (3x3), p(3x1)
# 출력: T (4x4)

def Rp2Trans(R, p):
    p = np.array(p).reshape(3, 1)
    T = np.block([[R,            p],
                  [np.zeros((1, 3)), 1]])
    return T


In [ ]:
# 3.8 동차 변환행렬 T로부터 회전행렬 R과 위치 벡터 p를 추출한다
# 입력: T 4x4
# 출력: R 3x3, p 3x1

def Trans2Rp(T):
  R = T[:3, :3]
  p = T[:3, 3]

  return R, p


In [ ]:
# 3.9 동차 변환행렬 T의 역행렬을 계산한다
# 입력: T (4x4)
# 출력: invT (4x4)
def TransInv(T):
    R, p = Trans2Rp(T)
    R_trans = RotInv(R)
    invT = np.block([[R_trans,           -np.dot(R_trans, p.reshape(3, 1))],
                     [np.zeros((1, 3)), 1]])
    return invT


In [ ]:
# 3.10 6차원 벡터인 twist에 해당하는 se(3) 행렬 계산
# 입력: V(twist) = [w v] 6x1 (w-> 3x1, v->3x1)
# 출력: [V] 4x4 행렬 형태

def Vec2se3(V):
    w = V[:3]
    v = V[3:]

    so3mat = Vec2so3(w) # w를 받아서 [w]를 반환 return so3mat (skew-symmetric 3x3 형태)

    se3mat = np.block([[so3mat,              v.reshape(3,1)],
                       [np.zeros((1, 3)),  0             ]])

    return se3mat


In [ ]:
# 3.11 se(3) 행렬 se3mat에 해당하는 6차원 벡터인 트위스트를 반환
# 입력: se3mat 4x4
# 출력: V      6x1

def se32Vec(se3mat):

    w = so32Vec(se3mat[:3, :3])
    v = se3mat[:3, 3]

    V = np.block([w, v])

    return V


In [ ]:
# 3.12 동차 변환행렬 T의 6x6 수반 표현 [Ad_T]를 계산
# 입력: T  4x4
# 출력 AdT 6x6

def Adjoint(T):
    R, p = Trans2Rp(T)
    # Vec2so3 함수 활용
    p_mat = Vec2so3(p)

    AdT = np.block([[R,                  np.zeros((3, 3))],
                   [np.dot(p_mat, R),   R               ]])

    return AdT


In [ ]:
# 3.13 점 q를 지나며 축의 방향이 단위 벡터 s이고 피치(선속력/각속력)이 h인 스크류 축에 해당하는 정규화된 스크류 축 S를 반환
# 입력: q, s, h (3x1, 3x1, 1)
# 출력: S 6x1

def Screw2Axis(q, s, h):
    w = np.array(s).flatten()
    v = np.cross(q, s) + h * w

    S = np.concatenate([w, v])

    return S


In [ ]:
# 3.14 6차원 지수 좌표 벡터 S * theta로부터 정규화된 스크류 축 S와 변위 theta를 추출
# 입력: expc6 6x1
# 출력: S 6x1, theta 1

def AxisAng(expc6):
    w = expc6[:3]
    v = expc6[3:]
    theta = np.linalg.norm(w)

    if theta == 0:
        theta = np.linalg.norm(v)

    S = expc6 / theta

    return S, theta


In [ ]:
# 3.15 se3mat의 행렬지수에 해당하는 동차 변환행렬 T->SE(3)를 계산
# 입력: se3mat 4x4
# 출력: T 4x4

def MatrixExp6(se3mat):
    so3mat = se3mat[:3, :3]
    w = so32Vec(so3mat)
    v = se3mat[:3, 3].reshape(3, 1)
    I = np.eye(3)

    theta = np.linalg.norm(w)

    if theta < 1e-6:
        T = np.block([[I, v],
                     [np.zeros((1, 3)), 1]])
    else:
        hat_w = w / theta
        w_mat = Vec2so3(hat_w)
        # MatrixExp3 함수 활용
        R = MatrixExp3(hat_w, theta)
        G = I * theta + (1 - np.cos(theta)) * w_mat + (theta - np.sin(theta)) * np.dot(w_mat, w_mat)
        p = np.dot(G, v / theta)

        T = np.block([[R, p],
                      [np.zeros((1, 3)), 1]])

    return T


In [ ]:
# 3.16 동차 변환행렬 T->SE(3)의 행렬 로그 se3mat를 계산
# 입력: T 4x4
# 출력: se3mat 4x4

def MatrixLog(T):
    R, p = Trans2Rp(T)
    p = p.reshape(3, 1)

    if np.linalg.norm(R - np.eye(3)) < 1e-6:
        se3mat = np.block([[np.zeros((3, 3)), p],
                           [np.zeros((1, 3)), 0]])
    else:
        theta = np.arccos(np.clip((np.trace(R) - 1) / 2.0, -1, 1))
        # MatrixLog3 함수 활용
        so3mat = MatrixLog3(R)
        w_mat = so3mat / theta
        I = np.eye(3)

        G_inv = (1.0/theta) * I - 0.5 * w_mat + (1.0/theta - 0.5 / np.tan(theta/2.0)) * np.dot(w_mat, w_mat)
        v = np.dot(G_inv, p)

        se3mat = np.block([[so3mat,   v],
                          [np.zeros((1, 3)), 0]])

    return se3mat
